# 02 — Cost weighting: ba giá trị lambda

Chạy độc lập bằng Run All sau khi cấu hình. Dùng chung `src/baf.py`; dữ liệu và kết quả nằm ngoài src. Train tháng0–5, validation6, không dự đoán tháng7. Giữ các notebook đã chạy trước đây trong project cũ.

LightGBM với w0=0.0114/0.057/0.285 và w1=1; ba seed, tổng9 lượt. Đây vẫn là class weighting, không phải chi phí tiền tệ riêng từng hồ sơ.

## 1. Môi trường và cấu hình

In [1]:
from pathlib import Path
import sys

PROJECT_DIR = None  # Ví dụ: r"D:\UIT\DS317\BAF_project_clean"
here = Path.cwd().resolve()
ROOT = (
    Path(PROJECT_DIR).resolve()
    if PROJECT_DIR
    else next(
        (path for path in [here, *here.parents] if (path / "src/baf.py").is_file()),
        None,
    )
)
if ROOT is None:
    raise FileNotFoundError("Mở project BAF_project_clean hoặc điền PROJECT_DIR.")

sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from IPython.display import display
import baf

if Path(baf.__file__).resolve() != (ROOT / "src/baf.py").resolve():
    raise RuntimeError(
        "Kernel đang giữ module của project khác. Restart Kernel trước khi Run All."
    )

print("Project:", ROOT)
print("Kernel:", sys.executable)
print(pd.Series(baf.package_versions()).to_string())


Project: D:\UIT\DS317\BAF_project
Kernel: c:\Users\ADMIN\AppData\Local\Python\pythoncore-3.14-64\python.exe
numpy            2.4.2
pandas           3.0.1
scipy           1.17.1
scikit-learn     1.7.2
psutil           7.2.2
lightgbm         4.7.0


In [2]:
DATA_PATH = ROOT / "data/raw/Base.csv"
previous_data = ROOT.parent / "BAF_project/data/raw/Base.csv"
if not DATA_PATH.is_file() and previous_data.is_file():
    DATA_PATH = previous_data

EXPECTED_ROWS = 1_000_000
DATASET_TAG = "BAF_Base"
SEEDS = [42, 317, 2026]
THREADS = 4
TREE_PARAMETERS = baf.TREE_PARAMETERS.copy()

METHODS = baf.COST_METHODS.copy()
RUN_DIR = baf.new_directory(ROOT / "results/runs", "cost")
print("Data:", DATA_PATH)
print("Methods:", METHODS)
print("Output:", RUN_DIR)


Data: D:\UIT\DS317\BAF_project\data\raw\Base.csv
Methods: ['lgbm_cost_0.0114', 'lgbm_cost_0.057', 'lgbm_cost_0.285']
Output: D:\UIT\DS317\BAF_project\results\runs\cost_20261027_101323_a07d5455


## 2. Kiểm tra dữ liệu và trọng số
Mọi phép fit tiền xử lý và tính trọng số chỉ dùng train.

In [3]:
frame = baf.load_data(DATA_PATH, expected_rows=EXPECTED_ROWS)
parts = baf.temporal_parts(frame)
display(pd.DataFrame(baf.profile(parts)).T)

weights_table = []
labels = parts["train"].fraud_bool.to_numpy()
for method in METHODS:
    weights = baf.sample_weights(labels, method)
    weights_table.append(
        {
            "method": method,
            "weight_negative": float(weights[labels == 0][0]),
            "weight_positive": float(weights[labels == 1][0]),
            "mean_weight": float(weights.mean()),
        }
    )
display(pd.DataFrame(weights_table))


,n,positive,months
train,794989,8151,"[0, 1, 2, 3, 4, 5]"
validation,108168,1450,[6]
test,96843,1428,[7]


,method,weight_negative,weight_positive,mean_weight
0,lgbm_cost_0.0114,0.0114,1.0,0.021536
1,lgbm_cost_0.057,0.0570,1.0,0.066669
2,lgbm_cost_0.285,0.2850,1.0,0.292331


## 3. Huấn luyện
Lượt mới luôn có thư mục mới. Log, cấu hình, source snapshot, metric và dự đoán được ghi tự động; lỗi được giữ lại.

In [4]:
metrics = baf.train_classical(
    data_path=DATA_PATH,
    output=RUN_DIR,
    methods=METHODS,
    seeds=SEEDS,
    threads=THREADS,
    tree_parameters=TREE_PARAMETERS,
    frame=frame,
    expected_rows=EXPECTED_ROWS,
    dataset_tag=DATASET_TAG,
)
display(metrics)


{
  "methods": [
    "lgbm_cost_0.0114",
    "lgbm_cost_0.057",
    "lgbm_cost_0.285"
  ],
  "seeds": [
    42,
    317,
    2026
  ],
  "threads": 4,
  "tree_parameters": {
    "objective": "binary",
    "class_weight": null,
    "n_estimators": 300,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_child_samples": 100,
    "subsample": 0.9,
    "subsample_freq": 1,
    "colsample_bytree": 0.9,
    "reg_lambda": 1.0,
    "verbosity": -1,
    "deterministic": true,
    "force_col_wise": true
  },
  "protocol": "temporal",
  "mode": "full",
  "evaluation_partitions": [
    "validation"
  ],
  "dataset_tag": "BAF_Base"
}
                 n positive              months
train       794989     8151  [0, 1, 2, 3, 4, 5]
validation  108168     1450                 [6]
test         96843     1428                 [7]
{"method": "lgbm_cost_0.0114", "seed": 42, "partition": "validation", "threshold": 0.7975021266490807, "fit_seconds": 30.203989399999955, "peak_rss_mib": 1360.5703125, "mean

,method,seed,partition,threshold,fit_seconds,peak_rss_mib,mean_weight,weight_negative,weight_positive,warnings,...,precision_frozen_threshold,alert_rate_frozen_threshold,recall_at_0p5,precision_at_0p5,f1_weighted_at_0p5,accuracy_at_0p5,tn,fp,fn,tp
0,lgbm_cost_0.0114,42,validation,0.797502,30.203989,1360.570312,0.021536,0.0114,1.0,,...,0.126098,0.055793,0.800690,0.060684,0.896077,0.831189,101444,5274,689,761
1,lgbm_cost_0.0114,317,validation,0.798572,28.159346,1367.753906,0.021536,0.0114,1.0,,...,0.126982,0.055987,0.797241,0.060359,0.895909,0.830911,101431,5287,681,769
2,lgbm_cost_0.0114,2026,validation,0.797755,28.871451,1367.015625,0.021536,0.0114,1.0,,...,0.125516,0.055978,0.794483,0.060132,0.895830,0.830782,101423,5295,690,760
3,lgbm_cost_0.057,42,validation,0.449033,31.535384,1369.519531,0.066669,0.0570,1.0,,...,0.124610,0.056311,0.488966,0.142427,0.966004,0.953683,101386,5332,691,759
4,lgbm_cost_0.057,317,validation,0.454297,30.738685,1369.957031,0.066669,0.0570,1.0,,...,0.124897,0.056181,0.487586,0.139338,0.965472,0.952759,101400,5318,691,759
5,lgbm_cost_0.057,2026,validation,0.449267,29.590118,1372.765625,0.066669,0.0570,1.0,,...,0.125696,0.056412,0.478621,0.139638,0.965843,0.953480,101383,5335,683,767
6,lgbm_cost_0.285,42,validation,0.139237,27.838587,1376.945312,0.292331,0.2850,1.0,,...,0.123231,0.056190,0.168276,0.305764,0.981394,0.983729,101389,5329,701,749
7,lgbm_cost_0.285,317,validation,0.140289,25.328383,1375.773438,0.292331,0.2850,1.0,,...,0.123534,0.055978,0.166207,0.307006,0.981408,0.983794,101411,5307,702,748
8,lgbm_cost_0.285,2026,validation,0.139251,22.945795,1373.406250,0.292331,0.2850,1.0,,...,0.124113,0.056015,0.166207,0.305450,0.981384,0.983757,101411,5307,698,752


## 4. Kiểm chứng và tổng hợp
Recall dùng thang0–1. Std là độ lệch chuẩn qua seed; một lượt thì để NaN, không tự điền0. Điểm validation không phải đánh giá độc lập ngoài mẫu.

In [5]:
report = baf.audit_results([RUN_DIR])
display(report.summary)
display(report.prediction_audit)
report.summary.to_csv(RUN_DIR / "summary.csv", index=False)


,method,partition,n_runs,ap_mean,ap_std,recall_mean,recall_std,fit_seconds_mean
0,lgbm_cost_0.0114,validation,3,0.181433,0.001620,0.526437,0.003402,29.078262
1,lgbm_cost_0.057,validation,3,0.181358,0.001275,0.525287,0.003185,30.621396
2,lgbm_cost_0.285,validation,3,0.179957,0.000504,0.517011,0.001436,25.370922


,method,seed,partition,sha256,verified
0,lgbm_cost_0.0114,42,validation,18c0ef53c5f76b413b9bd05cc0e69b1f89086a533aad63...,True
1,lgbm_cost_0.0114,317,validation,47cd40ec33c05a923e590e849967e00586b1300c5b4aa1...,True
2,lgbm_cost_0.0114,2026,validation,d772b16a3ea4debc59a0a18f274d63aa01b4dc8048b958...,True
3,lgbm_cost_0.057,42,validation,4aa7d28b06f6e142b99364f1f19d0c39b2cdc59da5c274...,True
4,lgbm_cost_0.057,317,validation,17d5335aaecb575c9ffde48f721cc1a911e7e9406ce036...,True
5,lgbm_cost_0.057,2026,validation,ec5bbffdcb9c0e43f26fab840cf1de3974ca5d428d56c4...,True
6,lgbm_cost_0.285,42,validation,bb7cfe160e17e2afc6658b228905db3bc3162003b64145...,True
7,lgbm_cost_0.285,317,validation,b323443f949a06eddf36148be69c7f7613e7f72ef90b6d...,True
8,lgbm_cost_0.285,2026,validation,4eeeade788f04ed3b685e0d66992b5ebb26212788131b8...,True


## 5. Xuất proof trong một lần
Hàm đọc lại predictions/log và thực thi notebook kiểm chứng mới. Không phụ thuộc Ctrl+S hoặc tên notebook đang mở. Lưu notebook đang mở khi kết thúc để giữ output giao diện; đó là thao tác riêng với việc tạo proof.

In [6]:
archive = baf.export_proof(
    run_paths=[RUN_DIR],
    output_root=ROOT / "results/proofs",
)


Verified prediction files: 9
Proof ZIP: D:\UIT\DS317\BAF_project\results\proofs\proof_20261027_101827_0d5aec05.zip
